In [2]:
from utils import * 
%load_ext autoreload
%autoreload 2

In [2]:
# forward_reads_path = '/groups/banfield/scratch/projects/environmental/sr/int/betazoid/rifle/sed_csp1_16ft.trimmed.PE.1.fastq.gz'
# reverse_reads_path = '/groups/banfield/scratch/projects/environmental/sr/int/betazoid/rifle/sed_csp1_16ft.trimmed.PE.2.fastq.gz'
# ref_path = '/home/philippar/curation/RifSed_csp1_16ft_4_scaffold_2135/scaffold_1.1.fasta'
# output_path = '/home/philippar/curation/RifSed_csp1_16ft_4_scaffold_2135/scaffold_1.1.bam'
# cmd = f'bbmap.sh pigz=t unpigz=t ambiguous=random minid=0.96 idfilter=0.97 threads=64 out=stdout.sam editfilter=5 in1={forward_reads_path} in2={reverse_reads_path} ref={ref_path} nodisk | shrinksam | sambam > {output_path}'
# print(cmd)

In [3]:
script = '''#!/bin/bash 

#SBATCH --job-name={sample_id}
#SBATCH --output={sample_id}.out
#SBATCH --cpus-per-task={num_threads}

cd {ncbi_output_dir}

mkdir -p {output_dir}
mkdir -p {tmp}
prefetch {srr_id} --max-size {max_size}

fasterq-dump {srr_id} --split-files -e {num_threads} -O {output_dir} --temp {tmp}
mv {fasterq_forward_reads_path} {forward_reads_path}
mv {fasterq_reverse_reads_path} {reverse_reads_path}

sickle pe -f {forward_reads_path} -r {reverse_reads_path} -t sanger -o {trimmed_forward_reads_path} -p {trimmed_reverse_reads_path} -s {singles_reads_path} -q {q} -l {l}
pigz -p {num_threads} {trimmed_forward_reads_path}
pigz -p {num_threads} {trimmed_reverse_reads_path}
'''



In [4]:
forward_reads_paths, reverse_reads_paths = dict(), dict()

SRR_ID = 'SRR6159086'
LOCATION = 'lake_superior_sediment'
BIOTITE_OUTPUT_DIR = '/groups/banfield/scratch/projects/environmental/sr/int/betazoid'
BIOTITE_NCBI_OUTPUT_DIR = os.path.join(BIOTITE_OUTPUT_DIR, 'ncbi') # Directory for the SRA cache thing that prefetch and fasterq-dump use.
BIOTITE_TMP_DIR = os.path.join(BIOTITE_OUTPUT_DIR, 'tmp')

PARAMS = dict()
PARAMS['srr_id'] = SRR_ID
PARAMS['sample_id'] = LOCATION
PARAMS['output_dir'] = BIOTITE_OUTPUT_DIR
PARAMS['ncbi_output_dir'] = BIOTITE_NCBI_OUTPUT_DIR
PARAMS['srr_path'] = os.path.join(BIOTITE_NCBI_OUTPUT_DIR, SRR_ID)
PARAMS['tmp'] = BIOTITE_TMP_DIR
PARAMS['forward_reads_path'] = os.path.join(PARAMS['output_dir'], f'{LOCATION}.PE.1.fastq')
PARAMS['reverse_reads_path'] = os.path.join(PARAMS['output_dir'], f'{LOCATION}.PE.2.fastq')
PARAMS['reads_path'] = os.path.join(PARAMS['output_dir'], f'{LOCATION}.fastq')
PARAMS['singles_reads_path'] = os.path.join(PARAMS['output_dir'], f'singles.fastq')
PARAMS['trimmed_forward_reads_path'] = os.path.join(PARAMS['output_dir'], f'{LOCATION}.trimmed.PE.1.fastq')
PARAMS['trimmed_reverse_reads_path'] = os.path.join(PARAMS['output_dir'], f'{LOCATION}.trimmed.PE.2.fastq')
PARAMS['fasterq_forward_reads_path'] = os.path.join(PARAMS['output_dir'], f'{SRR_ID}_1.fastq')
PARAMS['fasterq_reverse_reads_path'] = os.path.join(PARAMS['output_dir'], f'{SRR_ID}_2.fastq')
PARAMS['fasterq_reads_path'] = os.path.join(PARAMS['output_dir'], f'{SRR_ID}.fastq')
PARAMS['max_size'] = 4000000000
PARAMS['num_threads'] = 16
PARAMS['q'] = 20 
PARAMS['l'] = 50

print(script.format(**PARAMS))

print()
print(PARAMS['forward_reads_path'] + '\t' + PARAMS['reverse_reads_path'])

#!/bin/bash 

#SBATCH --job-name=lake_superior_sediment
#SBATCH --output=lake_superior_sediment.out
#SBATCH --cpus-per-task=16

cd /groups/banfield/scratch/projects/environmental/sr/int/betazoid/ncbi

mkdir -p /groups/banfield/scratch/projects/environmental/sr/int/betazoid
mkdir -p /groups/banfield/scratch/projects/environmental/sr/int/betazoid/tmp
prefetch SRR6159086 --max-size 4000000000

fasterq-dump SRR6159086 --split-files -e 16 -O /groups/banfield/scratch/projects/environmental/sr/int/betazoid --temp /groups/banfield/scratch/projects/environmental/sr/int/betazoid/tmp
mv /groups/banfield/scratch/projects/environmental/sr/int/betazoid/SRR6159086_1.fastq /groups/banfield/scratch/projects/environmental/sr/int/betazoid/lake_superior_sediment.PE.1.fastq
mv /groups/banfield/scratch/projects/environmental/sr/int/betazoid/SRR6159086_2.fastq /groups/banfield/scratch/projects/environmental/sr/int/betazoid/lake_superior_sediment.PE.2.fastq

sickle pe -f /groups/banfield/scratch/projects/envi

In [5]:
BIOTITE_OUTPUT_DIR = '/home/philippar/genes/annotation'

INTERPROSCAN_INPUT_PATH = '/home/philippar/genes/genes_untrimmed_unfiltered.faa'
INTERPROSCAN_OUTPUT_FILE_BASE = INTERPROSCAN_INPUT_PATH.replace('.faa', '')

cmd = f'interproscan.sh --output-file-base {INTERPROSCAN_OUTPUT_FILE_BASE} --input {INTERPROSCAN_INPUT_PATH} --formats TSV XML JSON'
print(cmd)

interproscan.sh --output-file-base /home/philippar/genes/genes_untrimmed_unfiltered --input /home/philippar/genes/genes_untrimmed_unfiltered.faa --formats TSV XML JSON


In [8]:
project_ids = ['SR-VP_9_9_2021_19_2A_1.5m', 'SR-VP_9_9_2021_56_4A_0.95m', 'SR-VP_9_9_2021_59_4A_0.85m', 'SR-VP_9_9_2021_31_2B_1.5m', 'SR-VP_9_9_2021_72_4B_1_05m_2', 'SR-VP_9_9_2021_52_3B_1.55m']
forward_reads_paths = [FORWARD_READS_PATHS[project_id] for project_id in project_ids]
reverse_reads_paths = [REVERSE_READS_PATHS[project_id] for project_id in project_ids]

MERGED_FORWARD_READS_PATH = '/home/philippar/serpens_ridge_soil.trimmed.PE.1.fastq.gz'
MERGED_REVERSE_READS_PATH = '/home/philippar/serpens_ridge_soil.trimmed.PE.2.fastq.gz'

# cat works for zipped files!
cmd = ' '.join(['cat'] + forward_reads_paths + ['>', MERGED_FORWARD_READS_PATH])
print(cmd)

cmd = ' '.join(['cat'] + reverse_reads_paths + ['>', MERGED_REVERSE_READS_PATH])
print(cmd)


cat /groups/banfield/sequences/2021/SR-VP_9_9_2021_19_2A_1.5m/raw.d/SR-VP_9_9_2021_19_2A_1.5m_trim_clean.PE.1.fastq.gz /groups/banfield/sequences/2021/SR-VP_9_9_2021_56_4A_0.95m/raw.d/SR-VP_9_9_2021_56_4A_0.95m_trim_clean.PE.1.fastq.gz /groups/banfield/sequences/2021/SR-VP_9_9_2021_59_4A_0.85m/raw.d/SR-VP_9_9_2021_59_4A_0.85m_trim_clean.PE.1.fastq.gz /groups/banfield/sequences/2021/SR-VP_9_9_2021_31_2B_1.5m/raw.d/SR-VP_9_9_2021_31_2B_1.5m_trim_clean.PE.1.fastq.gz /groups/banfield/sequences/2022/SR-VP_9_9_2021_72_4B_1_05m_2/raw.d/SR-VP_9_9_2021_72_4B_1_05m_2_trim_clean.PE.1.fastq.gz /groups/banfield/sequences/2021/SR-VP_9_9_2021_52_3B_1.55m/raw.d/SR-VP_9_9_2021_52_3B_1.55m_trim_clean.PE.1.fastq.gz > /home/philippar/serpens_ridge_soil.trimmed.PE.1.fastq.gz
cat /groups/banfield/sequences/2021/SR-VP_9_9_2021_19_2A_1.5m/raw.d/SR-VP_9_9_2021_19_2A_1.5m_trim_clean.PE.2.fastq.gz /groups/banfield/sequences/2021/SR-VP_9_9_2021_56_4A_0.95m/raw.d/SR-VP_9_9_2021_56_4A_0.95m_trim_clean.PE.2.fastq.gz

In [ ]:

BLASTN_DATABASE_DIR = '../data/blast/databases/'
BLASTN_DATABASE_NAME = 'ece'
BLASTN_DATABASE_PATH = os.path.join(BLASTN_DATABASE_DIR, BLASTN_DATABASE_NAME)

MAKEBLASTDB_PARAMS = dict()
MAKEBLASTDB_PARAMS['-in'] = '../data/ece.faa'
MAKEBLASTDB_PARAMS['-dbtype'] = 'prot'
MAKEBLASTDB_PARAMS['-title'] = BLASTN_DATABASE_NAME
MAKEBLASTDB_PARAMS['-out'] = BLASTN_DATABASE_PATH

BLASTN_OUTPUT_DIR = '../data/'

BLASTN_PARAMS = dict()
BLASTN_PARAMS['-db'] = BLASTN_DATABASE_PATH
BLASTN_PARAMS['-outfmt'] = '"6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore"'


if not os.path.exists(f'{BLASTN_DATABASE_PATH}.ndb'):
    cmd = ' '.join(['makeblastdb'] + [f'{param} {value}' for param, value in MAKEBLASTDB_PARAMS.items()])
    result = subprocess.run(cmd, shell=True, check=True, capture_output=True)
    if result.returncode == 0:
        print(f'Created BLAST database at {BLASTN_DATABASE_PATH}.')
    else:
        print(f'Failed to create BLAST database.')
else:
    print(f'Using existing BLAST database at {BLASTN_DATABASE_PATH}')

Created BLAST database at ../data/blast/databases/ece.


In [8]:
MMSEQS_TMP_DIR = '/home/prichter/Documents/banfield/betazoid/data/tmp'
MMSEQS_OUTPUT_PATH = os.path.join('/home/prichter/Documents/banfield/betazoid/data/genes/', 'genes') 
MMSEQS_INPUT_PATH = os.path.join('/home/prichter/Documents/banfield/betazoid/data/genes/', 'genes.faa')

MMSEQS_PARAMS = dict()
MMSEQS_PARAMS['--min-seq-id'] = 0.3
MMSEQS_PARAMS['--cov-mode'] = 5 # Short sequence-based coverage, i.e. coverage of alignment needs to be at least x% of the other sequence.
MMSEQS_PARAMS['-c'] = 0.8 

cmd = f'mmseqs easy-cluster {MMSEQS_INPUT_PATH} {MMSEQS_OUTPUT_PATH} {MMSEQS_TMP_DIR} ' + ' '.join([f'{param} {value}' for param, value in MMSEQS_PARAMS.items()])
print(cmd)
# subprocess.run(cmd, shell=True, check=True)


mmseqs easy-cluster /home/prichter/Documents/banfield/betazoid/data/genes/genes.faa /home/prichter/Documents/banfield/betazoid/data/genes/genes /home/prichter/Documents/banfield/betazoid/data/tmp --min-seq-id 0.3 --cov-mode 5 -c 0.8


In [ ]:
GENOME_CONTIG_IDS = dict()
GENOME_CONTIG_IDS['bz_0'] = []
GENOME_CONTIG_IDS['bz_1'] = ['Landfill_SRR22315504_scaffold_40513', 'Landfill_SRR22315504_scaffold_124901']
GENOME_CONTIG_IDS['bz_2'] = ['Dalian_Groundwater_SRR22387926_scaffold_5929']
GENOME_CONTIG_IDS['bz_3'] = ['Mangrove_60-80cm_incl_SRR17658263_scaffold_5020']
GENOME_CONTIG_IDS['bz_4'] = ['Nantong_Groundwater_SRR22387873_scaffold_1350']
GENOME_CONTIG_IDS['bz_5'] = ['Nantong_Groundwater_SRR22387873_scaffold_2044']
GENOME_CONTIG_IDS['bz_7'] = ['RifSed_csp1_16ft_4_scaffold_2135']
GENOME_CONTIG_IDS['bz_8'] = ['Shanghai_Industrial_Park_soil_SRR23875558_scaffold_1066']
GENOME_CONTIG_IDS['bz_9'] = ['SRR17498766_aquifer_sediment_scaffold_1816', 'SRR17498766_aquifer_sediment_scaffold_47226', 'SRR17498766_aquifer_sediment_scaffold_61411']
GENOME_CONTIG_IDS['bz_10'] = ['SRR17498766_aquifer_sediment_scaffold_6555', 'SRR17498766_aquifer_sediment_scaffold_7403', 'SRR17498766_aquifer_sediment_scaffold_93350']
GENOME_CONTIG_IDS['bz_11'] = ['SR-VP_9_9_2021_56_4A_0_95m_scaffold_3886', 'SR-VP_9_9_2021_31_2B_1_5m_scaffold_14874', 'SR-VP_9_9_2021_72_4B_1_05m_2_scaffold_578826', 'SR-VP_9_9_2021_56_4A_0_95m_scaffold_5485', 'SR-VP_9_9_2021_72_4B_1_05m_2_METASPADES_scaffold_91666']

GENOME_ID = 'bz_10'

CONTIG_METADATA_PATH = '../data/blast/ggkbase/blast_ggkbase_contig_metadata.csv' # This is NOT the clustered contigs, so there is redundancy. 
BIOTITE_OUTPUT_DIR = '/home/philippar/curation/'

contig_metadata_df = pd.read_csv(CONTIG_METADATA_PATH)
contig_df = FASTAFile.from_file('../data/blast/ggkbase/blast_ggkbase_contigs.fasta', filter_=lambda record : record.id in GENOME_CONTIG_IDS[GENOME_ID]).to_df()
contig_df['contig_id'] = contig_df.index 
contig_df = contig_df.drop_duplicates('contig_id').merge(contig_metadata_df, on='contig_id', how='left') # Merge contig sequences and metadata. 
contig_df.index = contig_df.contig_id


print(f'Metadata fields:', ', '.join(contig_metadata_df))
print(f'Number of entries:', len(contig_metadata_df))
print(f'Number of duplicate sequences:', contig_df.seq.duplicated().sum(), end='\n\n')


contig_df['project_id'] = contig_df.contig_id.map(PROJECT_IDS)
contig_df['forward_reads_path'] = contig_df.project_id.map(FORWARD_READS_PATHS)
contig_df['reverse_reads_path'] = contig_df.project_id.map(REVERSE_READS_PATHS)


print('Do all scaffolds have a reads path?', (contig_df.forward_reads_path.isnull().sum() == 0) and contig_df.reverse_reads_path.isnull().sum() == 0)

# Stringent parameters. 
BBMAP_PARAMS = dict()
BBMAP_PARAMS['local'] = 't'
BBMAP_PARAMS['pairedonly'] = 'f'
BBMAP_PARAMS['minid'] = 0.90
BBMAP_PARAMS['idfilter'] = 0.95

# Lenient parameters. 
BBMAP_PARAMS = dict()
BBMAP_PARAMS['local'] = 't'
BBMAP_PARAMS['pairedonly'] = 'f'
BBMAP_PARAMS['minid'] = 0.8
BBMAP_PARAMS['idfilter'] = 0.85

for row in contig_df.itertuples():

    output_dir = os.path.join(BIOTITE_OUTPUT_DIR, GENOME_ID)
    ref_path = os.path.join(BIOTITE_OUTPUT_DIR, GENOME_ID, 'scaffold_1.fasta')
    cmd, output_path = get_bbmap_command(ref_path, output_dir=output_dir, forward_reads_path=row.forward_reads_path, reverse_reads_path=row.reverse_reads_path, verbose=False, **BBMAP_PARAMS)
    print(cmd)